# Cinemática Radar Relativa (ARPA)
Cálculo analítico del CPA y TCPA frente a un buque en movimiento.

Este simulador está diseñado para fines educativos. **No lo utilices para la navegación real.**

<a href="https://colab.research.google.com/github/jorgejuan007/Nautica/blob/main/simulaciones/66_cinematica_radar_dos_barcos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

def simular_arpa(mi_rumbo, mi_vel, demora1, dist1, demora2, dist2, tiempo_minutos):
    # Convertir a radianes
    r_rad = np.radians(mi_rumbo)
    d1_rad = np.radians(demora1)
    d2_rad = np.radians(demora2)
    
    # Mi barco estático en el centro (movimiento relativo)
    # Posición 1 del otro barco
    x1 = dist1 * np.sin(d1_rad)
    y1 = dist1 * np.cos(d1_rad)
    
    # Posición 2 del otro barco
    x2 = dist2 * np.sin(d2_rad)
    y2 = dist2 * np.cos(d2_rad)
    
    # Vector de movimiento relativo en el tiempo dado
    dx = x2 - x1
    dy = y2 - y1
    
    # Velocidad relativa
    dist_relativa = np.hypot(dx, dy)
    vel_relativa = dist_relativa / (tiempo_minutos / 60)
    
    if vel_relativa == 0:
        print("El blanco está parado respecto a nosotros.")
        return
        
    # Rumbo Relativo
    rumbo_relativo = np.degrees(np.arctan2(dx, dy)) % 360
    
    # Calcular CPA (Closest Point of Approach)
    # Ecuación de la recta: pasa por (x2, y2) con vector (dx, dy)
    # Buscamos la distancia mínima al origen (0,0)
    # u = proyección escalar
    u = - (x2*dx + y2*dy) / (dx**2 + dy**2)
    
    if u < 0:
        print("El barco ya ha pasado el CPA (se está alejando).")
        cpa = np.hypot(x2, y2)
        tcpa_minutos = 0
    else:
        xcpa = x2 + u * dx
        ycpa = y2 + u * dy
        cpa = np.hypot(xcpa, ycpa)
        # TCPA: Tiempo para llegar al CPA desde la pos 2
        dist_al_cpa = np.hypot(xcpa - x2, ycpa - y2)
        tcpa_horas = dist_al_cpa / vel_relativa
        tcpa_minutos = tcpa_horas * 60
        
    print(f"--- Análisis ARPA (Cinemática Radar) ---")
    print(f"Velocidad Relativa del blanco: {vel_relativa:.1f} nudos a rumbo {rumbo_relativo:.1f}º")
    print(f"CPA (Punto de máxima aproximación): {cpa:.2f} Millas Náuticas")
    print(f"TCPA (Tiempo hasta el CPA): {tcpa_minutos:.1f} Minutos")
    
    if cpa < 1 and tcpa_minutos > 0:
        print("\n⚠️ ALERTA DE ABORDAJE: El blanco pasará a menos de 1 milla en breve.")
    
    # Gráfica
    fig, ax = plt.subplots(figsize=(6,6))
    ax.plot(0, 0, 'go', markersize=10, label='Mi Barco')
    ax.plot(x1, y1, 'bo', label='Blanco (t=0)')
    ax.plot(x2, y2, 'ro', label=f'Blanco (t={tiempo_minutos}m)')
    
    # Proyección relativa
    if u > 0:
        ax.plot([x2, x2 + u*dx*1.5], [y2, y2 + u*dy*1.5], 'r--', label='Trayectoria Relativa')
        ax.plot(xcpa, ycpa, 'kx', markersize=10, label='CPA')
        ax.plot([0, xcpa], [0, ycpa], 'k:')
    
    ax.set_aspect('equal')
    ax.grid(True)
    limit = max(dist1, dist2) * 1.2
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    plt.title('Cinemática Radar Relativa')
    plt.legend()
    plt.show()

# Widgets
mr = widgets.FloatSlider(value=0, min=0, max=359, description='Mi Rumbo:')
mv = widgets.FloatSlider(value=10, min=0, max=30, description='Mi Vel(Kts):')
d1 = widgets.FloatSlider(value=45, min=0, max=359, description='Demora 1:')
dt1 = widgets.FloatSlider(value=8, min=0.1, max=24, description='Dist 1 (nm):')
d2 = widgets.FloatSlider(value=44, min=0, max=359, description='Demora 2:')
dt2 = widgets.FloatSlider(value=6, min=0.1, max=24, description='Dist 2 (nm):')
t = widgets.FloatSlider(value=6, min=1, max=30, description='Tiempo(min):')

out = widgets.interactive_output(simular_arpa, {'mi_rumbo': mr, 'mi_vel': mv, 'demora1': d1, 'dist1': dt1, 'demora2': d2, 'dist2': dt2, 'tiempo_minutos': t})
display(widgets.VBox([widgets.HBox([mr, mv]), widgets.HBox([d1, dt1]), widgets.HBox([d2, dt2]), t, out]))
